[Guide Index](README.md)


# 03b SRL (SPARQL Rules)

StarLayer provides support for SRL (SPARQL Rules), a general-purpose Datalog-style rules language. SRL is specified by the W3C as SPARQL-RL, [SPARQL 1.2 RL](https://www.w3.org/TR/sparql12-rl/). 



## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.


In [1]:
from starlayer.graph import StarLayerGraph, Namespace

EX = Namespace("http://example.org/")


## 1. The base graph

The following graph is used in the SRL examples that follow.

A small software-dependency graph — the same running example [the SRL spec's §6.6 worked example](https://www.w3.org/TR/sparql12-rl/#worked-example) uses: 
    
a `:frontend` depends on an `:app`, which depends on a `:db` that has a known vulnerability; a `:logger` component has no vulnerabilities at all. The rules below will propagate the vulnerability along the dependency chain.

In [2]:
base = StarLayerGraph()
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .  # no vulnerability - stays safe in section 4

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")

base.bind("ex", EX)

print("data graph")
print(base.serialize())

data graph
@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:app ex:dependsOn ex:db ;
    a ex:Component .

ex:db ex:hasVulnerability ex:vuln1 ;
    a ex:Component .

ex:frontend ex:dependsOn ex:app ;
    a ex:Component .

ex:logger a ex:Component .

ex:vuln1 ex:severity 9.1 .



## 2. Write and parse SRL rules

SRL rules follow a simple structure:  `RULE { head } WHERE { body }`

SRL rules work like a SPARQL `CONSTRUCT`: the `WHERE` pattern matches triples in the graph, and the head says which new triples to create. Two things are different. A rule can match triples that other rules created, so rules can build on each other. A rule can even build on its own output. Because of this, SRL runs the whole rule set repeatedly, adding the new triples each time, until a pass produces nothing new.

The two rules below show both. R1 creates `ex:exposedTo` triples from `ex:hasVulnerability`. R2 matches its own output: `ex:exposedTo` spreads through any number of `ex:dependsOn` hops.

`srl.parse_ruleset()` parses SRL text into a `srl.RuleSet`, a plain Python dataclass tree. `srl.ruleset_to_text()` renders the ruleset back to SRL text.

In [3]:
from starlayer.sparql import srl

rules_text = """
PREFIX ex: <http://example.org/>

# R1: a component is exposed to any vulnerability it has directly.
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }

# R2: exposure propagates along dependsOn (recursive - depends on itself).
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }
"""

ruleset = srl.parse_ruleset(rules_text)
print(f"{len(ruleset.rules)} rules parsed")
print(srl.ruleset_to_text(ruleset, namespace_manager=base.namespace_manager))


2 rules parsed
RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }


## 3. Run inference

`ruleset.infer(base_graph)` evaluates the rule set and returns a *new* graph containing only the newly-inferred triples. It runs repeated passes until no new triples appear.

In this example the database has a vulnerability. R1 finds that the database is exposed. On the next pass R2 finds that the application, which depends on the database, is exposed. On the pass after that R2 finds that the front-end, which depends on the application, is exposed. A further pass produces nothing new, so evaluation stops.

In [4]:
# inferred is the set of new triples.
inferred = ruleset.infer(base)
inferred.bind("ex", EX)
print(inferred.serialize())


@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 .

ex:db ex:exposedTo ex:vuln1 .

ex:frontend ex:exposedTo ex:vuln1 .



## 4. `FILTER` and negation

Two more rules: R3 flags a component as critically exposed if any of its vulnerabilities has `severity >= 9.0` (`FILTER`); R4 marks a component safe if it's *not* critically exposed (`NOT { }`, negation-as-failure).

R4 genuinely depends on R3 having finished. SRL's stratification algorithm works this out from the rules' structure, so every component's exposure is fully propagated *before* R4 runs.

In [5]:

rules_text += """
# R3: severity >= 9.0 counts as a critical vulnerability.
RULE { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }

# R4: not critically exposed -> safe to deploy.
RULE { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }
"""

ruleset = srl.parse_ruleset(rules_text)
inferred = ruleset.infer(base)
inferred.bind("ex", EX)
print()
print("inferred graph")
print(inferred.serialize(format="turtle"))



inferred graph
@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:db ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:frontend ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:logger ex:status ex:safeToDeploy .




## 5. Seeing the evaluation plan

Before running anything, SRL creates a plan that describes how the rules will be revalauted.  `ruleset.plan()` provide visibility to the plan: the **strata** in evaluation order, and the **dependency edges** between rules that the strata were worked out from. Rules are identified by their position in `ruleset.rules`, so R1 is index 0.

An edge `source -> target` means the `source` rule's body reads something the `target` rule's head can produce. It is **open** for an ordinary dependency, and **closed** when the source reads the target through `NOT { }` (or is a rule that must see its inputs complete). A closed edge forces the source into a *later* stratum than the target.

For the four rules from section 4, R4 has a closed edge to R3, so it waits for the stratum after R3:

In [6]:
plan = ruleset.plan()

for stratum in plan.strata:
    print(f"stratum {stratum.number}:", ", ".join(f"R{r.index + 1}" for r in stratum.general))

for edge in sorted(plan.edges, key=lambda e: (e.source, e.target)):
    print(f"R{edge.source + 1} -> R{edge.target + 1} ({edge.label})")

assert [[r.index for r in s.general] for s in plan.strata] == [[0, 1, 2], [3]]
assert {(e.source, e.target): e.label for e in plan.edges}[(3, 2)] == "closed"

stratum 0: R1, R2, R3
stratum 1: R4
R2 -> R1 (open)
R2 -> R2 (open)
R3 -> R1 (open)
R3 -> R2 (open)
R4 -> R3 (closed)


### Rule order doesn't matter

The plan comes from the rules' *structure*, not from the order they are written in. Below, R4 is written and listed before R3, and each rule is given an id so the plan can name it. R4 still lands in the later stratum, and `infer()` produces exactly the same result as section 4.

In [7]:
reordered_rules_text = """
PREFIX ex: <http://example.org/>

RULE ex:R1 { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE ex:R2 { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

# R4 listed before R3 on purpose - it has a closed dependency on R3 through NOT { }.
RULE ex:R4 { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }

RULE ex:R3 { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }
"""

reordered_ruleset = srl.parse_ruleset(reordered_rules_text)

for stratum in reordered_ruleset.plan().strata:
    print(f"stratum {stratum.number}:", ", ".join(base.qname(r.id) for r in stratum.general))

reordered_inferred = reordered_ruleset.infer(base)
print("same result regardless of rule order:", set(reordered_inferred) == set(inferred))
assert set(reordered_inferred) == set(inferred)

stratum 0: ex:R1, ex:R2, ex:R3
stratum 1: ex:R4
same result regardless of rule order: True


## 6. When no order exists

Some rule sets have no valid evaluation order. A rule that matches `NOT { }` of something it produces itself can never be settled: it would need its own output to be complete before it could run. SRL's §4.4.1 leaves this undefined, so StarLayer raises `StratificationError`, from `plan()` and from `infer()`.

This is also why a rule set cannot run forever. A rule that only reuses terms already in the graph can only ever produce a finite number of triples, so the repeated passes must stop. The only rules that could invent new values are those with an assignment (`SET(...)`) or a blank node in the head. Those run once per stratum, and a cycle through one is rejected in the same way.

In [8]:
from starlayer.sparql.srl_eval import StratificationError

unstratifiable = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:flagged ex:yes } WHERE { ?x a ex:Component . NOT { ?x ex:flagged ex:yes } }
""")

for attempt in (unstratifiable.plan, lambda: unstratifiable.infer(base)):
    try:
        attempt()
    except StratificationError as e:
        print("StratificationError:", e)

StratificationError: SRL: stratification condition violated - a recursive dependency involves a closed edge, so this rule set has no well-defined evaluation outcome per §4.4.1
StratificationError: SRL: stratification condition violated - a recursive dependency involves a closed edge, so this rule set has no well-defined evaluation outcome per §4.4.1


## 7. Asking a question with `query()`

`ruleset.query(base_graph, goal_pattern)` allows for a triple pattern query against the result of running rules against a data graph. Each match is a dictionary from variable to value. Here, which components are critically exposed?

In [9]:
from rdflib import Variable

goal = srl.TriplePattern(subject=Variable("x"), predicate=EX.status, object=EX.criticallyExposed)

for row in ruleset.query(base, goal):
    print(base.qname(row[Variable("x")]))

ex:frontend
ex:db
ex:app


## 8. Catching a malformed rule before evaluating it

SRL's own [§4.2 Well-formedness Conditions](https://www.w3.org/TR/sparql12-rl/#well-formedness-conditions) - every variable used in a rule's head must actually be bound by its own body - are checked automatically *inside* `srl.parse_ruleset()`. A rule set that violates this raises `SRLParseError` at parse time, rather than parsing and silently producing a wrong answer later. This is an easy mistake to make by hand, for example with a typo'd variable name.

After that check, `parse_ruleset()` also validates the rule set against StarLayer's `srl:` SHACL shapes (it encodes the rule set as RDF to do so; see [SRL (SPARQL Rules) as RDF](05c-srl-sparql-rules-as-rdf.ipynb)) and raises `SRLParseError` if it does not conform. This takes about a tenth of a second, so a program that parses many rule sets it already trusts can pass `validate=False` to skip it.

In [10]:
try:
    broken = srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ?x ex:flaggedBy ?reviewer } WHERE { ?x ex:hasVulnerability ?v . }
    """)
except srl.SRLParseError as e:
    print("SRLParseError:", e)


SRLParseError: SRL: rule set is not well-formed per §4.2: unbound_head_variable (reviewer in rule head)


## 9. `DATA` blocks

A rule set can carry its own ground facts in a `DATA { }` block.  The facts travel with the rules and get combined with the base graph before the rules are run. 

A rule matches them exactly as it matches triples from the base graph, and they also appear in the output.

The spec names three graphs, and this section depends on telling them apart:
- the **base graph** is the data you pass to `infer()`;
- the **evaluation graph** is what the rules match as they run: the base graph, plus the rule set's `DATA { }` facts, plus every triple inferred so far;
- the **inference graph** is what `infer()` returns: the inferred triples, not including anything already in the base graph.

A `DATA { }` block behaves like a rule with an empty body: its triples join the evaluation graph without any rule being run.

Here the rule set records which team owns two of the components, and one rule derives a `reportsTo` triple from each:

In [11]:
# `base` is the software-dependency graph from section 1
team_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

DATA { ex:db ex:team ex:platform . ex:app ex:team ex:web . }

RULE { ?x ex:reportsTo ?t } WHERE { ?x ex:team ?t }
""")

team_inferred = team_rules.infer(base)
team_inferred.bind("ex", EX)
print(team_inferred.serialize())

assert (EX.db, EX.team, EX.platform) in team_inferred      # the DATA facts are in the output
assert (EX.db, EX.reportsTo, EX.platform) in team_inferred # and the rule matched them

@prefix ex: <http://example.org/> .

ex:app ex:reportsTo ex:web ;
    ex:team ex:web .

ex:db ex:reportsTo ex:platform ;
    ex:team ex:platform .



### `WHERE DATA` and `NOT DATA`: matching base data

A rule normally matches the whole evaluation graph. Writing `DATA` after `WHERE` (or after a `NOT`) changes that to **base data**: the rule then matches only the base graph.

The keyword is easy to misread. `WHERE DATA` does **not** mean "match the `DATA { }` block". The base graph is the data you passed to `infer()`, and it is separate from the rule set's own `DATA { }` facts.

| A rule's `WHERE` matches... | plain `WHERE { }` | `WHERE DATA { }` |
|---|---|---|
| the base graph passed to `infer()` | yes | yes |
| the rule set's own `DATA { }` facts | yes | **no** |
| triples inferred by the rules so far | yes | **no** |

Use it for a rule that should look only at the data you supplied, never at the rule set's own facts or at other rules' conclusions. The spec's section "Matching Base Data" (§3.9) says the same: matching on the base graph "does not include triples added by rule set evaluation from the rule set DATA blocks".

The same `DATA` fact is used below by three rules:
- the first uses a plain `WHERE`, so it matches the `DATA` fact;
- the second uses `WHERE DATA` for the same pattern, so it matches nothing, because `ex:team` exists only in the `DATA` block, not in the base graph;
- the third uses `WHERE DATA` for `ex:hasVulnerability`, which *is* in the base graph, so it matches.

In [12]:
# `base` is the software-dependency graph from section 1
scope_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

DATA { ex:db ex:team ex:platform . }

RULE { ?x ex:seenInData ?t }  WHERE      { ?x ex:team ?t }
RULE { ?x ex:seenInBase ?t }  WHERE DATA { ?x ex:team ?t }
RULE { ?x ex:flagged ex:yes } WHERE DATA { ?x ex:hasVulnerability ?v }
""")

scope_inferred = scope_rules.infer(base)
scope_inferred.bind("ex", EX)
print(scope_inferred.serialize())

assert (EX.db, EX.seenInData, EX.platform) in scope_inferred       # ordinary WHERE sees the DATA fact
assert not list(scope_inferred.triples((None, EX.seenInBase, None)))  # WHERE DATA does not
assert (EX.db, EX.flagged, EX.yes) in scope_inferred               # WHERE DATA sees the base graph

@prefix ex: <http://example.org/> .

ex:db ex:flagged ex:yes ;
    ex:seenInData ex:platform ;
    ex:team ex:platform .



### `NOT DATA`: filling in a default

`DATA` can also be used on a single `NOT`. The spec's own use is a **default value**: derive a value only when the base graph does not already state one. The rule below gives every vulnerable component a review priority unless the base graph already has one. The cell builds its own small base graph (just `ex:db`, with a vulnerability) and runs the rule three ways: with no priority in the base graph, with one already there, and with a plain `NOT` instead of `NOT DATA`.

The rule negates the very predicate it produces (`ex:reviewPriority`). With a plain `NOT { }` that is an unstratifiable cycle, because the rule would need its own output to be finished before it could run. `NOT DATA { }` looks only at the base graph, so there is no cycle, and the rest of the rule body can still match inferred triples.

In [13]:
# The base graph for this example: ex:db is a component with a known vulnerability.
# (Section 1's graph is not used here, so the example stands on its own.)
review_base = StarLayerGraph()
review_base.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:db a ex:Component ; ex:hasVulnerability ex:vuln1 .
""", format="turtle")

# The rule: a component with a vulnerability gets a high review priority,
# unless the base graph already states a priority for it.
default_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:reviewPriority ex:high }
WHERE { ?x ex:hasVulnerability ?v . NOT DATA { ?x ex:reviewPriority ?existing } }
""")

# Case 1: the base graph has no priority for ex:db, so the default is derived.
case1 = default_rules.infer(review_base)
case1.bind("ex", EX)
print("Case 1 - base graph has no priority. Derived:")
print(case1.serialize())

# Case 2: the base graph already states a priority for ex:db, so nothing is derived.
review_base_with_priority = StarLayerGraph()
review_base_with_priority += review_base
review_base_with_priority.add((EX.db, EX.reviewPriority, EX.low))
case2 = default_rules.infer(review_base_with_priority)
print("Case 2 - base graph already has ex:db ex:reviewPriority ex:low. Triples derived:", len(case2))

assert (EX.db, EX.reviewPriority, EX.high) in case1
assert len(case2) == 0

# Case 3: the same rule with a plain NOT (matching the whole evaluation graph) cannot be ordered.
plain_not = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:reviewPriority ex:high }
WHERE { ?x ex:hasVulnerability ?v . NOT { ?x ex:reviewPriority ?existing } }
""")
try:
    plain_not.plan()
except StratificationError as e:
    print("Case 3 - the same rule with a plain NOT cannot be ordered:", type(e).__name__)

Case 1 - base graph has no priority. Derived:
@prefix ex: <http://example.org/> .

ex:db ex:reviewPriority ex:high .

Case 2 - base graph already has ex:db ex:reviewPriority ex:low. Triples derived: 0


Case 3 - the same rule with a plain NOT cannot be ordered: StratificationError


## 10. Triple terms and annotations (RDF 1.2)

A **triple term** `<<( s p o )>>` is a whole triple used as a value, so a rule can talk about a claim without asserting it. Rule patterns, rule heads and `DATA` blocks can all contain them, and the variables inside one are ordinary rule variables. SRL understands the `turtle12` annotation similar to SPARQL 1.2.

Here Alice believes that Bob is 130 years old and Carol believes that Dan is 40. The rule matches the claim inside the triple term, with `?person` and `?age` bound from within it, and records who holds an implausible belief:

In [14]:
claims = StarLayerGraph()
claims.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:believes <<( ex:bob ex:age 130 )>> .
    ex:carol ex:believes <<( ex:dan ex:age 40 )>> .
""", format="turtle12")

doubt_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?who ex:doubts ?person }
WHERE { ?who ex:believes <<( ?person ex:age ?age )>> . FILTER(?age > 120) }
""")

doubts = doubt_rules.infer(claims)
doubts.bind("ex", EX)
print(doubts.serialize())

assert (EX.alice, EX.doubts, EX.bob) in doubts
assert len(doubts) == 1

@prefix ex: <http://example.org/> .

ex:alice ex:doubts ex:bob .



A rule **head** can build a triple term too. The variables in the head's triple term are filled in from the rule body. The rule below records that a review disputes each implausible claim: it builds the claim itself, `<<( ex:bob ex:age 130 )>>`, as the object of a new triple, without asserting that Bob is 130:

In [15]:
dispute_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ex:review ex:disputes <<( ?person ex:age ?age )>> }
WHERE { ?who ex:believes <<( ?person ex:age ?age )>> . FILTER(?age > 120) }
""")

disputes = dispute_rules.infer(claims)
disputes.bind("ex", EX)
print(disputes.serialize())

from rdflib import Literal
from starlayer.graph.model.triple import TripleTerm

assert (EX.review, EX.disputes, TripleTerm(EX.bob, EX.age, Literal(130))) in disputes
assert len(disputes) == 1

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:review ex:disputes <<( ex:bob ex:age 130 )>> .



Various ways of writing a triple that has been annotated work as they do in SPARQL 1.2: an **annotation block** `{| ... |}`, a **reifier** `~ ?r`, and a **reified triple** `<< s p o >>`. Each one stands for a triple, a reifier for it, and the statements made about the reifier.

In this example the claims carry a certainty. The rule keeps only those at least half certain, matching the certainty through the annotation block:

In [16]:
annotated = StarLayerGraph()
annotated.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:says ex:bobIsOld {| ex:certainty 0.9 |} .
    ex:alice ex:says ex:bobIsTall {| ex:certainty 0.2 |} .
""", format="turtle12")

strong_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?who ex:strongClaim ?claim }
WHERE { ?who ex:says ?claim {| ex:certainty ?c |} . FILTER(?c > 0.5) }
""")

strong = strong_rules.infer(annotated)
strong.bind("ex", EX)
print(strong.serialize())

assert (EX.alice, EX.strongClaim, EX.bobIsOld) in strong
assert len(strong) == 1

@prefix ex: <http://example.org/> .

ex:alice ex:strongClaim ex:bobIsOld .



## 11. Property paths

A rule body can use a **property path** as a predicate, to follow several steps in one pattern. SRL allows just two path operators:
- `a/b` follows `a` and then `b`;
- `^a` follows `a` backwards.

Parentheses group them, as in `^(a/b)`. Paths are for rule bodies, including inside a `NOT`. A rule head and a `DATA` block take plain predicates only.

Using the dependency graph from section 1, the first rule finds a component that reaches a vulnerability in two steps, and the second turns a vulnerability back into the component that has it:

In [17]:
# `base` is the software-dependency graph from section 1
path_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:affectedVia ?v } WHERE { ?x ex:dependsOn/ex:hasVulnerability ?v }
RULE { ?v ex:affects ?x }     WHERE { ?v ^ex:hasVulnerability ?x }
""")

path_inferred = path_rules.infer(base)
path_inferred.bind("ex", EX)
print(path_inferred.serialize())

print("patterns in the first rule's body:", len(path_rules.rules[0].body))

assert (EX.app, EX.affectedVia, EX.vuln1) in path_inferred   # app depends on db, which has vuln1
assert (EX.vuln1, EX.affects, EX.db) in path_inferred
assert len(path_rules.rules[0].body) == 2

@prefix ex: <http://example.org/> .

ex:app ex:affectedVia ex:vuln1 .

ex:vuln1 ex:affects ex:db .

patterns in the first rule's body: 2


Any other path operator (`|`, `*`, `+`, `?`, `!`) is rejected when the rule set is parsed. The spec leaves them out on purpose: a step that repeats is written as a recursive rule instead, like `ex:exposedTo` in section 2, and an alternative is written as two rules.

In [18]:
for path in ("ex:dependsOn+", "ex:dependsOn|ex:hasVulnerability"):
    try:
        srl.parse_ruleset(f"""
        PREFIX ex: <http://example.org/>
        RULE {{ ?x ex:reaches ?y }} WHERE {{ ?x {path} ?y }}
        """)
    except srl.SRLParseError as e:
        print(f"{path}:", e)

ex:dependsOn+: SRL: property path <http://example.org/dependsOn>+ is not allowed: SRL property paths use only the / and ^ operators (write a repeated or optional step with a recursive rule instead)
ex:dependsOn|ex:hasVulnerability: SRL: property path <http://example.org/dependsOn>|<http://example.org/hasVulnerability> is not allowed: SRL property paths use only the / and ^ operators (write a repeated or optional step with a recursive rule instead)


## 12. Importing rule sets

A rule set can import other rule sets with `IMPORTS <iri>`, so rules can be kept in shared libraries. Importing means reading a document, which has security implications, so StarLayer never reads one unless you say how. There are three steps:

1. **Have the libraries as SRL text.** They can come from anywhere: files, a verified copy, a database. Here they are Python strings.
2. **Write a loader.** A loader is a function that takes the IRI of an import and returns that library's text. It is the one place that decides where libraries come from.
3. **Pass the loader to `infer()`** (or `query()` or `plan()`) as `imports_loader=`.

The imports are resolved before any rule runs, each time you call `infer()`. A document is read **once** per call, even if several rule sets import it or the imports form a cycle, and the imported rules and `DATA` are merged into the rule set.

Here a base library defines the exposure rule, a second library builds on it, and our rule set imports the second one:

In [ ]:
# Step 1: the libraries, as SRL text
base_library = """
PREFIX ex: <http://example.org/>
RULE ex:directExposure { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v }
"""

exposure_library = """
PREFIX ex: <http://example.org/>
IMPORTS <http://example.org/lib/base.srl>
RULE ex:inheritedExposure { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v }
"""

libraries = {
    "http://example.org/lib/base.srl": base_library,
    "http://example.org/lib/exposure.srl": exposure_library,
}


# Step 2: the loader - given the IRI of an import, returns that library's text
def library_loader(iri):
    return libraries[iri]


# The rule set to run: it imports the exposure library and adds one rule of its own
with_imports = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
IMPORTS <http://example.org/lib/exposure.srl>
RULE ex:markCritical { ?x ex:status ex:criticallyExposed } WHERE { ?x ex:exposedTo ?v }
""")

# Step 3: pass the loader to infer()
imported = with_imports.infer(base, imports_loader=library_loader)
imported.bind("ex", EX)


print(imported.serialize())

assert (EX.frontend, EX.status, EX.criticallyExposed) in imported

rule plan
@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:db ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:frontend ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .



To see what the imports contributed, `srl.resolve_imports()` returns the rule set with the imported rules merged in and no imports left. Resolving once and running the result many times avoids reading the libraries on every `infer()`:

In [20]:
resolved = srl.resolve_imports(with_imports, library_loader)
print(f"{len(with_imports.rules)} rule written, {len(resolved.rules)} after resolving the imports; imports left: {resolved.imports}")

# the resolved rule set needs no loader
assert len(resolved.infer(base)) == len(imported)
assert len(resolved.rules) == 3

1 rule written, 3 after resolving the imports; imports left: []


`plan()` takes the same `imports_loader=`, so you can see how the rules will run *after* the imports are resolved. Each rule here has an id so the plan can name it. The plan lists the two imported rules, `ex:directExposure` and `ex:inheritedExposure`, together with the rule written in this rule set, `ex:markCritical`, and the edges show which depends on which. All three share one stratum because none of them uses `NOT`; a dependency only forces a later stratum when it is through a `NOT`.

In [21]:
plan = with_imports.plan(imports_loader=library_loader)

names = {r.index: base.qname(r.id) for stratum in plan.strata for r in stratum.general + stratum.run_once}

for stratum in plan.strata:
    print(f"stratum {stratum.number}:", ", ".join(names[r.index] for r in stratum.general))
for edge in sorted(plan.edges, key=lambda e: (e.source, e.target)):
    print(f"{names[edge.source]} depends on {names[edge.target]} ({edge.label})")

assert set(names.values()) == {"ex:markCritical", "ex:directExposure", "ex:inheritedExposure"}

stratum 0: ex:markCritical, ex:inheritedExposure, ex:directExposure
ex:markCritical depends on ex:inheritedExposure (open)
ex:markCritical depends on ex:directExposure (open)
ex:inheritedExposure depends on ex:inheritedExposure (open)
ex:inheritedExposure depends on ex:directExposure (open)


## 13. SRL as a syntax tree

A parsed rule set can also be written as an RDF syntax tree; see [SRL (SPARQL Rules) as RDF](05c-srl-sparql-rules-as-rdf.ipynb).

## 14. Not yet covered

These parts of SRL are not in StarLayer yet, or not finished in the specification. This guide will gain a section for each when they are:

- **Aggregation** (`GROUP BY`, `COUNT`, `COUNT DISTINCT`, `MIN`, `MAX`, `SUM`). The draft describes an aggregation rule element, but it is still an open issue (Issue 840) with no grammar yet. Until then an aggregate inside `FILTER` or `SET` is rejected when the rule set is parsed.
- **Translating a rule set to SPARQL.** A placeholder appendix in the draft (Issue 1303).
- **Choosing among valid evaluation orders.** A rule set can have more than one valid stratification; `plan()` shows the one StarLayer used.